# Module 2 Overview

This notebook demonstrates the NLP preprocessing pipeline for the Internal Helpdesk Chatbot (Module 2).
It loads the Module 1 dataset, applies text normalization, tokenization, stopword removal, lemmatization,
intent and entity normalization, and generates NLP-ready outputs.


# Load Module 1 Dataset

We start by loading the processed dataset from Module 1.


In [1]:
import sys
import os
from pathlib import Path

# Get the current working directory and find the project root
# When running in notebook, we can use the current directory
current_dir = Path.cwd()
# Navigate to the project root (assuming we're in the notebooks directory)
project_root = current_dir.parent

# Ensure the src directory is on the path
SRC_DIR = project_root / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nlp_preprocessor import NLPPreprocessor, REQUIRED_COLUMNS

# Initialize the preprocessor
preprocessor = NLPPreprocessor(project_root)

# Load the Module 1 dataset
input_csv = project_root / 'data' / 'processed' / 'faq_dataset.csv'
df = preprocessor.load_dataset(input_csv)

print(f"Loaded {len(df)} records.")
df.head()

Loaded 294 records.


,question,intent,answer,entity
0,How do I reset my password?,password_reset,You can reset your password through the intern...,password
1,I forgot my company password.,password_reset,Use the 'Forgot Password' link on the login pa...,password
2,Where can I change my password?,password_reset,You can change your password by going to Setti...,password
3,I cannot remember my login password.,password_reset,Click 'Forgot Password' on the login screen an...,password
4,How do I create a new password?,password_reset,"Log into the employee portal, navigate to Sett...",password


# Dataset Validation

Validate the Module 1 dataset before preprocessing.


In [2]:
# Check for missing values and required columns
missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"Required columns present: {REQUIRED_COLUMNS}")
print(f"Number of records: {len(df)}")
print(f"Number of unique intents: {df['intent'].nunique()}")
print(f"Minimum examples per intent: {df['intent'].value_counts().min()}")

# Check for missing values in required columns
for col in REQUIRED_COLUMNS:
    missing = df[col].isna().sum() + (df[col].astype(str).str.strip() == '').sum()
    print(f"Missing values in '{col}': {missing}")

Required columns present: ['question', 'intent', 'answer', 'entity']
Number of records: 294
Number of unique intents: 22
Minimum examples per intent: 12
Missing values in 'question': 0
Missing values in 'intent': 0
Missing values in 'answer': 0
Missing values in 'entity': 0


# Text Normalization

We demonstrate the `clean_text` function which normalizes raw text.


In [3]:
# Example of text cleaning
examples = [
    "  How Do I Reset My Company Password??? ",
    "What's the Wi-Fi password?!",
    "I can't access my email!!!"
]

for ex in examples:
    cleaned = preprocessor.clean_text(ex)
    print(f"Original: {ex!r}")
    print(f"Cleaned:  {cleaned!r}\n")

Original: '  How Do I Reset My Company Password??? '
Cleaned:  'how do i reset my company password'

Original: "What's the Wi-Fi password?!"
Cleaned:  "what's the wi-fi password"

Original: "I can't access my email!!!"
Cleaned:  "i can't access my email"



# Tokenization

We demonstrate tokenization using the `tokenize_text` function.


In [4]:
# Example of tokenization
cleaned_examples = [
    "how do i reset my company password",
    "what's the wi-fi password",
    "i cannot access my email"
]

for ex in cleaned_examples:
    tokens = preprocessor.tokenize_text(ex)
    print(f"Text: {ex}")
    print(f"Tokens: {tokens}\n")

Text: how do i reset my company password
Tokens: ['how', 'do', 'i', 'reset', 'my', 'company', 'password']

Text: what's the wi-fi password
Tokens: ['what', 'is', 'the', 'password']

Text: i cannot access my email
Tokens: ['i', 'can', 'not', 'access', 'my', 'email']



# Stopword Analysis

We demonstrate stopword removal using the `remove_stopwords` function.
Note: the original cleaned question is preserved, and stopwords are removed only for `filtered_tokens`.


In [5]:
# Example of stopword removal
token_lists = [
    ['how', 'do', 'i', 'reset', 'my', 'password'],
    ['what', 'is', 'the', 'wi', 'fi', 'password'],
    ['i', 'cannot', 'access', 'my', 'email']
]

for tokens in token_lists:
    filtered = preprocessor.remove_stopwords(tokens)
    print(f"Original tokens: {tokens}")
    print(f"After stopword removal: {filtered}\n")

Original tokens: ['how', 'do', 'i', 'reset', 'my', 'password']
After stopword removal: ['reset', 'password']

Original tokens: ['what', 'is', 'the', 'wi', 'fi', 'password']
After stopword removal: ['wi', 'fi', 'password']

Original tokens: ['i', 'cannot', 'access', 'my', 'email']
After stopword removal: ['cannot', 'access', 'email']



# Lemmatization

We demonstrate lemmatization using the `lemmatize_tokens` function.
Note: the POS tagger is used when available for verb lemmatization (e.g., 'running' -> 'run').


In [6]:
# Example of lemmatization
token_lists = [
    ['employees', 'running', 'issues'],
    ['reset', 'passwords'],
    ['connecting', 'to', 'the', 'wi', 'fi']
]

for tokens in token_lists:
    lemmatized = preprocessor.lemmatize_tokens(tokens)
    print(f"Original tokens: {tokens}")
    print(f"Lemmatized tokens: {lemmatized}\n")

Original tokens: ['employees', 'running', 'issues']
Lemmatized tokens: ['employee', 'run', 'issue']

Original tokens: ['reset', 'passwords']
Lemmatized tokens: ['reset', 'password']

Original tokens: ['connecting', 'to', 'the', 'wi', 'fi']
Lemmatized tokens: ['connect', 'to', 'the', 'wi', 'fi']



# Intent Normalization

We demonstrate intent normalization using the `normalize_intent` function.


In [7]:
# Example of intent normalization
intent_examples = [
    "Password_Reset",
    "password_reset ",
    " PASSWORD_RESET ",
    "Email_Problems",
    "  eMail_PrObLeMs  "
]

for intent in intent_examples:
    normalized = preprocessor.normalize_intent(intent)
    print(f"Original: {intent!r} -> Normalized: {normalized!r}")

Original: 'Password_Reset' -> Normalized: 'password_reset'
Original: 'password_reset ' -> Normalized: 'password_reset'
Original: ' PASSWORD_RESET ' -> Normalized: 'password_reset'
Original: 'Email_Problems' -> Normalized: 'email_problems'
Original: '  eMail_PrObLeMs  ' -> Normalized: 'email_problems'


# Entity Preparation

We demonstrate entity normalization using the `normalize_entity` function.


In [8]:
# Example of entity normalization
entity_examples = [
    "password_reset",
    "wifi_problem",
    "laptop_support",
    "email_problems",
    "salary_information"
]

for entity in entity_examples:
    normalized = preprocessor.normalize_entity(entity)
    print(f"Original: {entity!r} -> Normalized: {normalized!r}")

Original: 'password_reset' -> Normalized: 'password'
Original: 'wifi_problem' -> Normalized: 'wifi'
Original: 'laptop_support' -> Normalized: 'laptop'
Original: 'email_problems' -> Normalized: 'email'
Original: 'salary_information' -> Normalized: 'salary'


# Entity Extraction

We demonstrate rule-based entity extraction using the `extract_entities` function.


In [9]:
# Example of entity extraction
question_examples = [
    "How do I reset my password?",
    "My laptop cannot connect to Wi-Fi.",
    "When will my salary be credited?",
    "What is the contact number for IT support?",
    "I need help with my email account."
]

for question in question_examples:
    cleaned = preprocessor.clean_text(question)
    entities = preprocessor.extract_entities(cleaned)
    print(f"Question: {question}")
    print(f"Cleaned:  {cleaned}")
    print(f"Entities: {entities}\n")

Question: How do I reset my password?
Cleaned:  how do i reset my password
Entities: ['password']

Question: My laptop cannot connect to Wi-Fi.
Cleaned:  my laptop cannot connect to wi-fi
Entities: ['laptop', 'wifi']

Question: When will my salary be credited?
Cleaned:  when will my salary be credited
Entities: ['salary']

Question: What is the contact number for IT support?
Cleaned:  what is the contact number for it support
Entities: ['contact', 'it_support']

Question: I need help with my email account.
Cleaned:  i need help with my email account
Entities: ['email', 'account', 'help']



# Before vs After Examples

We show real examples from the dataset after preprocessing.


In [10]:
# Preprocess the entire dataset
df_processed = preprocessor.preprocess_dataset(df)

# Select a few examples from different intents
intents_shown = set()
examples = []

for _, row in df_processed.iterrows():
    intent = row['intent']
    if intent not in intents_shown:
        intents_shown.add(intent)
        examples.append(row)
    if len(intents_shown) >= 5:
        break

print("--- BEFORE / AFTER EXAMPLES ---\n")

for row in examples:
    print("Original:")
    print(" ", row['question'])
    print("Clean:")
    print(" ", row['clean_question'])
    print("Tokens:")
    print(" ", row['tokens'])
    print("Filtered Tokens:")
    print(" ", row['filtered_tokens'])
    print("Lemmatized Tokens:")
    print(" ", row['lemmatized_tokens'])
    print("Intent:")
    print(" ", row['intent'])
    print("Entity:")
    print(" ", row['entity'])
    print()

--- BEFORE / AFTER EXAMPLES ---

Original:
  How do I reset my password?
Clean:
  how do i reset my password
Tokens:
  ['how', 'do', 'i', 'reset', 'my', 'password']
Filtered Tokens:
  ['reset', 'password']
Lemmatized Tokens:
  ['reset', 'password']
Intent:
  password_reset
Entity:
  password

Original:
  How do I recover my account?
Clean:
  how do i recover my account
Tokens:
  ['how', 'do', 'i', 'recover', 'my', 'account']
Filtered Tokens:
  ['recover', 'account']
Lemmatized Tokens:
  ['recover', 'account']
Intent:
  account_access
Entity:
  account

Original:
  I need help with my laptop.
Clean:
  i need help with my laptop
Tokens:
  ['i', 'need', 'help', 'with', 'my', 'laptop']
Filtered Tokens:
  ['need', 'help', 'laptop']
Lemmatized Tokens:
  ['need', 'help', 'laptop']
Intent:
  laptop_problems
Entity:
  laptop

Original:
  How do I install new software?
Clean:
  how do i install new software
Tokens:
  ['how', 'do', 'i', 'install', 'new', 'software']
Filtered Tokens:
  ['install',

# NLP Statistics

We generate statistics from the preprocessed dataset.


In [11]:
# Generate statistics
stats = preprocessor.generate_nlp_statistics(df_processed)

print(f"Total questions: {stats['total_questions']}")
print(f"Total intents: {stats['total_intents']}")
print(f"Total entities: {stats['unique_entities']}")
print(f"Average question length: {stats['avg_question_length']}")
print(f"Average token count: {stats['avg_token_count']}")
print(f"Raw vocabulary size: {stats['raw_vocabulary_size']}")
print(f"Lemmatized vocabulary size: {stats['lemmatized_vocabulary_size']}")
print(f"Average stopwords removed: {stats['avg_stopwords_removed']}")

print("\nIntent distribution:")
for intent, count in sorted(stats['intent_distribution'].items(), key=lambda x: (-x[1], x[0])):
    print(f"  {intent}: {count}")

print("\nEntity frequency:")
for entity, count in sorted(stats['entity_frequency'].items(), key=lambda x: (-x[1], x[0])):
    print(f"  {entity}: {count}")

Total questions: 294
Total intents: 22
Total entities: 22
Average question length: 33.1
Average token count: 6.51
Raw vocabulary size: 470
Lemmatized vocabulary size: 386
Average stopwords removed: 3.47

Intent distribution:
  help: 20
  password_reset: 20
  laptop_problems: 19
  software_installation: 17
  email_problems: 14
  account_access: 12
  attendance: 12
  contact_information: 12
  employee_id: 12
  goodbye: 12
  greetings: 12
  holidays: 12
  hr_support: 12
  internet_problems: 12
  leave_policy: 12
  office_location: 12
  payroll: 12
  salary_information: 12
  security: 12
  technical_support: 12
  wifi_problems: 12
  working_hours: 12

Entity frequency:
  help: 20
  password: 20
  laptop: 19
  software: 17
  email: 14
  account: 12
  attendance: 12
  contact: 12
  employee_id: 12
  goodbye: 12
  greeting: 12
  holiday: 12
  hr: 12
  internet: 12
  it_support: 12
  leave: 12
  location: 12
  payroll: 12
  salary: 12
  security: 12
  wifi: 12
  working_hours: 12


# Visualizations

We generate the required charts: token length distribution, entity frequency, and question length distribution.


In [12]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use('Agg')  # Use non-interactive backend

# Token length distribution
token_counts = df_processed['tokens'].apply(len)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(token_counts, bins=range(0, int(token_counts.max()) + 2), color='#4C72B0', edgecolor='white')
ax.set_xlabel('Number of tokens per question')
ax.set_ylabel('Number of questions')
ax.set_title('Token Length Distribution')
fig.tight_layout()

chart1_path = project_root / 'outputs' / 'charts' / 'token_length_distribution.png'
chart1_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(chart1_path, dpi=120)
plt.close(fig)
print(f"Saved token length distribution chart to: {chart1_path}")

# Entity frequency
entities = df_processed['entity'].astype(str).str.strip()
entity_counts = entities[entities != ""].value_counts()
fig, ax = plt.subplots(figsize=(9, 6))
entity_counts.plot(kind='bar', ax=ax, color='#55A868')
ax.set_xlabel('Entity')
ax.set_ylabel('Frequency')
ax.set_title('Entity Frequency')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()

chart2_path = project_root / 'outputs' / 'charts' / 'entity_frequency.png'
fig.savefig(chart2_path, dpi=120)
plt.close(fig)
print(f"Saved entity frequency chart to: {chart2_path}")

# Question length distribution
question_lens = df_processed['clean_question'].astype(str).str.len()
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(question_lens, bins=30, color='#C44E52', edgecolor='white')
ax.set_xlabel('Number of characters in cleaned question')
ax.set_ylabel('Number of questions')
ax.set_title('Question Length Distribution')
fig.tight_layout()

chart3_path = project_root / 'outputs' / 'charts' / 'question_length_distribution.png'
fig.savefig(chart3_path, dpi=120)
plt.close(fig)
print(f"Saved question length distribution chart to: {chart3_path}")

Saved token length distribution chart to: D:\New Project\CodeVedex Projects\CODEVEDX\Project-04-AI-Helpdesk-Chatbot\outputs\charts\token_length_distribution.png
Saved entity frequency chart to: D:\New Project\CodeVedex Projects\CODEVEDX\Project-04-AI-Helpdesk-Chatbot\outputs\charts\entity_frequency.png


Saved question length distribution chart to: D:\New Project\CodeVedex Projects\CODEVEDX\Project-04-AI-Helpdesk-Chatbot\outputs\charts\question_length_distribution.png


# NLP Output Generation

We save the NLP-ready dataset as CSV and JSON.


In [13]:
# Save the NLP-ready dataset
output_dir = project_root / 'data' / 'processed'
csv_path, json_path = preprocessor.save_nlp_dataset(df_processed, output_dir)

print(f"Saved CSV to: {csv_path}")
print(f"Saved JSON to: {json_path}")

# Verify the files exist and have content
print(f"CSV file exists: {csv_path.exists()}")
print(f"JSON file exists: {json_path.exists()}")
print(f"CSV file size: {csv_path.stat().st_size} bytes")
print(f"JSON file size: {json_path.stat().st_size} bytes")

Saved CSV to: D:\New Project\CodeVedex Projects\CODEVEDX\Project-04-AI-Helpdesk-Chatbot\data\processed\faq_nlp_ready.csv
Saved JSON to: D:\New Project\CodeVedex Projects\CODEVEDX\Project-04-AI-Helpdesk-Chatbot\data\processed\faq_nlp_ready.json
CSV file exists: True
JSON file exists: True
CSV file size: 88064 bytes
JSON file size: 193831 bytes


# Final Validation

We run the final quality gate checks to ensure everything is correct.


In [14]:
# Validate the preprocessed data
validation = preprocessor.validate_preprocessed_data(df_processed)

if validation['is_valid']:
    print("Preprocessing validation: PASS")
else:
    print("Preprocessing validation: FAIL")
    for error in validation['errors']:
        print(f"  - {error}")
    for warning in validation['warnings']:
        print(f"  - {warning}")

# Compute the quality gate
report = preprocessor.compute_quality_gate(
    df,  # input_df
    df_processed,  # output_df
    validation,
    stats,
    artifacts={
        "csv": csv_path.exists(),
        "json": json_path.exists(),
        "report": True,  # We'll generate the report below
        "charts": all([
            chart1_path.exists(),
            chart2_path.exists(),
            chart3_path.exists()
        ])
    }
)

print("\nQuality Gate Checks:")
for check, passed in report['checks'].items():
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {check}")

print(f"\nOverall: {'PASS' if report['overall_pass'] else 'FAIL'}")

Preprocessing validation: PASS

Quality Gate Checks:
[PASS] Module 1 dataset loads
[PASS] Input records = 294
[PASS] Output records = 294
[PASS] Record count consistency
[PASS] Intent count remains 22
[PASS] Minimum intent examples >= 12
[PASS] clean_question generated
[PASS] Tokenization works
[PASS] Stopword processing works
[PASS] Lemmatization works
[PASS] Intent normalization works
[PASS] Entity preparation works
[PASS] Entity extraction works
[PASS] No unintended data loss
[PASS] No duplicate questions introduced
[PASS] No missing required values
[PASS] NLP-ready CSV generated
[PASS] NLP-ready JSON generated
[PASS] NLP report generated
[PASS] NLP charts generated
[PASS] Preprocessing validation passes

Overall: PASS


# Module 2 Quality Gate

If all checks pass, Module 2 is complete.
